# nh3 smoke test

Rust-backed HTML sanitizer (Python binding to the Ammonia Rust crate).

In [ ]:
# Example 1: Default clean() removes dangerous tags (script) but keeps benign markup.
import nh3

dirty = "<b>safe</b><script>alert('xss')</script><p>hello</p>"
cleaned = nh3.clean(dirty)
print(f"input:  {dirty!r}")
print(f"output: {cleaned!r}")

assert "<script>" not in cleaned
assert "alert" not in cleaned
assert "<b>safe</b>" in cleaned
assert "<p>hello</p>" in cleaned

In [ ]:
# Example 2: Restrict allowed tags via the `tags` parameter.
import nh3

dirty = "<b>bold</b> and <i>italic</i> and <a href='http://example.com'>link</a>"

# Only allow <b>; strip <i> and <a>.
cleaned = nh3.clean(dirty, tags={"b"})
print(f"cleaned with tags={{'b'}}: {cleaned!r}")

assert "<b>bold</b>" in cleaned
assert "<i>" not in cleaned
assert "<a " not in cleaned
# Text content of stripped tags is preserved.
assert "italic" in cleaned
assert "link" in cleaned

In [ ]:
# Example 3: clean_text() escapes content for safe text-context insertion.
import nh3

raw = "<script>alert('xss')</script> & some text"
escaped = nh3.clean_text(raw)
print(f"input:  {raw!r}")
print(f"output: {escaped!r}")

# clean_text escapes all HTML metacharacters; output should contain no raw <, >.
assert "<" not in escaped
assert ">" not in escaped
assert "&" in escaped  # `&` must be escaped to `&amp;`

In [ ]:
# Example 4: is_html() detects HTML-like content.
import nh3

assert nh3.is_html("<p>hello</p>") is True
assert nh3.is_html("<b>bold</b> text") is True
assert nh3.is_html("just plain text") is False
assert nh3.is_html("") is False

print("is_html('<p>hello</p>'):   ", nh3.is_html("<p>hello</p>"))
print("is_html('just plain text'):", nh3.is_html("just plain text"))

In [ ]:
# Example 5: Per-tag attribute allow-list via the `attributes` parameter.
import nh3

dirty = '<a href="http://example.com" onclick="steal()" title="ok">link</a>'

# Allow <a> tag with only href and title; drop onclick.
cleaned = nh3.clean(
    dirty,
    tags={"a"},
    attributes={"a": {"href", "title"}},
)
print(f"cleaned: {cleaned!r}")

assert 'href="http://example.com"' in cleaned
assert 'title="ok"' in cleaned
assert "onclick" not in cleaned
assert "steal" not in cleaned
print("\nAll nh3 smoke checks passed.")